# ch00 · STREAM triad on a free Colab GPU (C0.3 starter)

Runtime → Change runtime type → **T4 GPU**, then run all cells.
T4 spec-sheet bandwidth: **320 GB/s**. How close do you get?

In [ ]:
import torch
assert torch.cuda.is_available(), 'switch the runtime to a GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
import time

n = 50_000_000
b = torch.rand(n, device='cuda', dtype=torch.float64)
c = torch.rand(n, device='cuda', dtype=torch.float64)
a = torch.zeros(n, device='cuda', dtype=torch.float64)

def triad():
    torch.add(b, c, alpha=2.5, out=a)  # fused a = b + 2.5*c, no temporaries

for _ in range(3):
    triad()                      # warm-up (ch00 rule 1)
torch.cuda.synchronize()         # ch00 rule 3: never time async work unsynced

times = []
for _ in range(10):
    t0 = time.perf_counter()
    triad()
    torch.cuda.synchronize()
    times.append(time.perf_counter() - t0)

med = sorted(times)[len(times) // 2]   # ch00 rule 2: median, not mean
gbs = 3 * n * 8 / med / 1e9
print(f'STREAM triad: {gbs:.0f} GB/s  ({gbs / 320:.0%} of T4 spec)')

Record your number in `results/parallel_result.md`. Bonus: rerun with
`dtype=torch.float32` — does GB/s change? Should it?